# 05 — Classifier-Free Guidance

CFG enables conditional generation by training a single model to handle both conditional and unconditional generation.

## CFG Mechanics and the Guidance Scale Tradeoff

Classifier guidance (Dhariwal & Nichol, 2021) combined a diffusion model with a separately trained classifier to steer generation. The score is modified as:
$$\tilde{\nabla}_{x_t} \log p = \nabla_{x_t} \log p(x_t) + w \nabla_{x_t} \log p(y|x_t)$$

**Classifier-free guidance** (Ho & Salimans, 2022) eliminates the classifier entirely. The model is trained to optionally condition on a class label *y*; during training, *y* is randomly dropped out (set to ∅ with probability ~10-20%). At inference:
$$\tilde{\epsilon}_\theta(x_t, y) = (1+w)\epsilon_\theta(x_t, y) - w\epsilon_\theta(x_t, \emptyset)$$

The guidance scale *w* controls the tradeoff:
- *w=0*: unconditional generation
- *w=1*: standard conditional generation
- *w>1*: overconditioning — higher fidelity to the class, lower diversity

This is the mechanism used in Stable Diffusion and DALL-E: large w (e.g., 7.5) produces sharp, prompt-consistent images at the cost of sample diversity.

In [ ]:
# CFG conditional generation demo
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(42)

# Two-class 2D dataset
def sample_data(n=2000):
    half = n // 2
    x0 = torch.randn(half, 2) * 0.4 + torch.tensor([-2.0, 0.0])  # class 0
    x1 = torch.randn(half, 2) * 0.4 + torch.tensor([2.0, 0.0])   # class 1
    X = torch.cat([x0, x1])
    y = torch.cat([torch.zeros(half, dtype=torch.long), torch.ones(half, dtype=torch.long)])
    return X, y

data, labels = sample_data()

T = 100
betas = torch.linspace(1e-4, 0.02, T)
alphas = 1 - betas
alpha_bars = torch.cumprod(alphas, dim=0)
N_CLASSES = 2
NULL_CLASS = N_CLASSES  # dropout token

class CFGNoiseNet(nn.Module):
    def __init__(self, data_dim=2, hidden=64, T=100, n_classes=3):
        super().__init__()
        self.time_embed = nn.Embedding(T, hidden)
        self.class_embed = nn.Embedding(n_classes, hidden)  # includes null token
        self.net = nn.Sequential(
            nn.Linear(data_dim + hidden*2, hidden), nn.GELU(),
            nn.Linear(hidden, hidden), nn.GELU(),
            nn.Linear(hidden, data_dim)
        )

    def forward(self, x, t, c):
        t_e = self.time_embed(t)
        c_e = self.class_embed(c)
        h = torch.cat([x, t_e, c_e], dim=-1)
        return self.net(h)

cfg_net = CFGNoiseNet(T=T, n_classes=3)
opt = torch.optim.Adam(cfg_net.parameters(), lr=3e-4)

# Training with label dropout
for step in range(3000):
    idx = torch.randint(0, len(data), (256,))
    x0 = data[idx]
    c = labels[idx].clone()
    t = torch.randint(0, T, (256,))
    noise = torch.randn_like(x0)
    x_t = alpha_bars[t].sqrt().unsqueeze(1) * x0 + (1-alpha_bars[t]).sqrt().unsqueeze(1) * noise

    # Dropout labels 15% of the time
    mask = torch.rand(256) < 0.15
    c[mask] = NULL_CLASS

    eps_pred = cfg_net(x_t, t, c)
    loss = ((eps_pred - noise)**2).mean()
    opt.zero_grad(); loss.backward(); opt.step()

print('CFG model trained')

In [ ]:
# CFG sampling at different guidance scales
@torch.no_grad()
def cfg_sample(model, n_samples, target_class, guidance_scale, T, alpha_bars, alphas, betas):
    x = torch.randn(n_samples, 2)
    c_cond = torch.full((n_samples,), target_class, dtype=torch.long)
    c_uncond = torch.full((n_samples,), NULL_CLASS, dtype=torch.long)

    for t in range(T - 1, -1, -1):
        t_batch = torch.full((n_samples,), t, dtype=torch.long)
        eps_cond = model(x, t_batch, c_cond)
        eps_uncond = model(x, t_batch, c_uncond)
        # CFG linear combination
        eps = (1 + guidance_scale) * eps_cond - guidance_scale * eps_uncond

        mean = (1/alphas[t].sqrt()) * (x - betas[t]/(1-alpha_bars[t]).sqrt() * eps)
        if t > 0:
            x = mean + betas[t].sqrt() * torch.randn_like(x)
        else:
            x = mean
    return x

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
guidance_scales = [0.0, 0.5, 1.0, 3.0]

for class_label, row_axes in enumerate([axes[0], axes[1]]):
    for scale, ax in zip(guidance_scales, row_axes):
        samples = cfg_sample(cfg_net, 200, class_label, scale, T, alpha_bars, alphas, betas)
        ax.scatter(samples[:, 0].numpy(), samples[:, 1].numpy(), alpha=0.3, s=8)
        ax.set_xlim(-5, 5); ax.set_ylim(-3, 3)
        ax.set_title(f'Class {class_label}, w={scale}')

plt.suptitle('CFG: Guidance Scale Effect on Conditional Generation')
plt.tight_layout()
plt.savefig('/tmp/cfg_guidance.png', dpi=100, bbox_inches='tight')
plt.show()
print('CFG guidance comparison saved')